# TN1 — LSTM hai chiều ở ngân sách nhỏ

## Câu hỏi

TN1 cho kết quả: LSTM hơn cả hai biến thể TCN, và thu nhỏ LSTM 96% gần như
không mất gì.

| cấu hình | tham số | cv_score | GHIJ |
|---|---|---|---|
| LSTM-352 | 1.502.713 | 0.7570 ± 0.0041 | 0.8103 ± 0.0154 |
| LSTM-67 | 56.908 | 0.7532 ± 0.0020 | 0.8017 ± 0.0025 |
| DS-TCN-64 | 56.281 | 0.7421 ± 0.0007 | 0.7958 ± 0.0154 |
| TCN-64 | 151.513 | 0.7423 ± 0.0044 | chưa chạy |

Cơ chế hồi quy hợp bài toán này. Câu hỏi tiếp theo:

> **Đọc cửa sổ theo cả hai chiều có tốt hơn một chiều không?**

## Vì sao hai chiều là hợp lệ

Model nhận **200 mẫu quá khứ**, đoán **25 mẫu tiếp theo**. Cả 200 mẫu vào đều
có sẵn lúc dự báo, nên đọc chúng theo chiều nào cũng được. "Tương lai" cần đoán
là mẫu 201–225, model không hề thấy — **không có rò rỉ**.

Bản của MobiVital chỉ lấy trạng thái ở bước cuối, nên thông tin từ mẫu thứ 1
phải sống sót qua 200 bước cổng mới tới được đầu ra. Đọc thêm chiều ngược thì
đầu ra thấy được cả hai đầu cửa sổ.

MobiVital không thử hướng này.

## Vì sao hidden 41

Hai chiều làm tăng gấp bội tham số. Không hạ hidden xuống thì so không công bằng:

```
LSTM-67       một chiều   hidden 67    56.908 tham số
BiLSTM-41     hai chiều   hidden 41    57.507 tham số     lệch 1,1%
BiLSTM-67     hai chiều   hidden 67   149.703 tham số     to gấp 2,6 lần
```

Lấy hidden 41 thì thắng thua chỉ còn một nguyên nhân: **chiều đọc**. Nếu lấy
hidden 67 mà thắng thì không biết thắng vì hai chiều hay vì to hơn — đúng cái
bẫy mà mục LSTM-67 vừa gỡ được cho TCN.

Đặt cạnh **LSTM-67** để so, không phải LSTM-352.

## Giới hạn

Bộ tham số huấn luyện lấy từ `checkpoints/optimal_params.json` của MobiVital,
được dò cho `hidden=352` một chiều. BiLSTM-41 chạy bằng bộ đó, chưa được dò
riêng.

Điều kiện này áp dụng như nhau cho LSTM-67 và DS-TCN-64, nên phép so ở cùng
ngân sách tham số vẫn có giá trị. Nhưng không kết luận được về tiềm năng của
BiLSTM nếu được dò tham số riêng.

## 1. Chuẩn bị Colab

Mount Drive để lấy cửa sổ train đã cắt ở `DATA_PREPARE.ipynb`.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

Tải mã nguồn rồi vào thư mục đó. Xem dòng `commit đồ án` để chắc đang chạy bản mới.

In [ ]:
# Xoá trước để chạy lại ô này luôn lấy mã mới nhất, không dính bản cũ.
!rm -rf /content/UWB_RADAR
!git clone -q https://github.com/quangminhho004-blip/UWB_RADAR.git /content/UWB_RADAR
%cd /content/UWB_RADAR
!python scripts/setup_colab.py

Lấy `by_user/` và `windows/` từ Drive. Không cần CSV thô 13 GB.

In [ ]:
!python scripts/restore_processed_data_on_drive.py

## 2. Kiểm bản cài đặt

Chỗ dễ sai nhất của BiLSTM: lấy đặc trưng bằng `output[:, -1, :]` thay vì
`h[-2], h[-1]`.

Với LSTM hai chiều, `output` có dạng `(batch, 200, 2*hidden)`:

```
output[:, t, :hidden]    chiều xuôi tại bước t
output[:, t, hidden:]    chiều ngược tại bước t
```

Lấy `output[:, -1, :]` thì nửa đầu là chiều xuôi **đã đọc hết 200 mẫu** — đúng.
Nhưng nửa sau là chiều ngược **mới đọc đúng một mẫu**, vì với chiều ngược thì
bước cuối chính là bước đầu tiên nó xử lý.

Sai chỗ đó thì model vẫn chạy, vẫn ra số, nhưng nửa thông tin gần như trống —
rồi kết luận *"BiLSTM không giúp"* trong khi thật ra code hỏng.

`scripts/check_model.py` chạy tám phép kiểm, trong đó phép số 6 **chứng minh hai
cách cho kết quả khác nhau**, để không ai lặp lại lỗi này. Phép số 8 kiểm số tham
số có xấp xỉ LSTM-67 không — lệch quá 5% thì dừng, vì không so công bằng được.

Sai bất kỳ phép nào là script dừng hẳn, không chạy tiếp.

In [ ]:
!python scripts/check_model.py --model bilstm --hidden 41 --compare-with lstm --compare-hidden 67


## 3. BiLSTM-41 — 4 fold CV, 3 seed

Bảy phép kiểm đạt thì mới chạy.

Cấu hình huấn luyện giữ y nguyên như bốn cấu hình trước: 20 epoch, Adam lr 1e-4,
batch 64, MSE, `corr` 0.9, bốn fold cũ. Chỉ đổi kiến trúc.

Sau mỗi fold script tự nén rồi chép sang Drive, tên chứa cấu hình nên không đè
tệp nào. Ngắt phiên giữa chừng thì chạy lại ô này, fold đã xong được bỏ qua.

Khoảng **1 giờ**.

In [ ]:
!python scripts/run_cv.py --experiment tn1 --model bilstm --hidden 41 --seed 0
!python scripts/run_cv.py --experiment tn1 --model bilstm --hidden 41 --seed 1
!python scripts/run_cv.py --experiment tn1 --model bilstm --hidden 41 --seed 2

## 4. Cất kết quả

`run_cv.py` đã tự nén sau mỗi fold. Ô dưới nén lại một lần sau khi xong cả 12
lần chạy, ra tên riêng.

Bảng so đủ các cấu hình dựng ở `TN1_final_evaluation.ipynb` — phiên này chỉ có
dòng của riêng nó trong `summary.csv`.

In [ ]:
!python scripts/save_results.py tn1 --out tn1_bilstm_h41

## 5. Ngắt phiên

Colab giữ runtime sau khi ô cuối chạy xong và vẫn tính giờ. Ô này đóng phiên
lại. Kết quả đã nén sang Drive ở mục 4 nên ngắt ở đây không mất gì.

Bấm liên tiếp các ô mục 3, 4, 5 thì Colab xếp hàng chạy lần lượt, không cần
ngồi canh.

In [ ]:
from google.colab import runtime
runtime.unassign()